# Analise arquivos raw contas-pagar.csv

In [1]:
import os
import pandas as pd
from pandasql import sqldf

pysqldf = lambda q: sqldf(q, globals())

In [2]:
BASE_DIR = os.path.dirname(os.path.dirname(os.path.dirname(os.path.abspath('.'))))
DATA_DIR = os.path.join(BASE_DIR, 'data')
RAW_DIR =  os.path.join(DATA_DIR, 'raw')

In [4]:
df = pd.read_csv(os.path.join(RAW_DIR, 'contas_pagar.csv'), sep=';')
df.shape

(404, 10)

## Analise exploratória

In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 404 entries, 0 to 403
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   id_titulo_pagar    404 non-null    str    
 1   id_fornecedor      397 non-null    str    
 2   data_emissao       404 non-null    str    
 3   data_vencimento    404 non-null    str    
 4   data_pagamento     253 non-null    str    
 5   categoria_despesa  396 non-null    str    
 6   valor_titulo       397 non-null    float64
 7   valor_pago         397 non-null    float64
 8   status             397 non-null    str    
 9   forma_pagamento    397 non-null    str    
dtypes: float64(2), str(8)
memory usage: 31.7 KB


In [6]:
df.isna().sum()

id_titulo_pagar        0
id_fornecedor          7
data_emissao           0
data_vencimento        0
data_pagamento       151
categoria_despesa      8
valor_titulo           7
valor_pago             7
status                 7
forma_pagamento        7
dtype: int64

In [7]:
df.head()

,id_titulo_pagar,id_fornecedor,data_emissao,data_vencimento,data_pagamento,categoria_despesa,valor_titulo,valor_pago,status,forma_pagamento
0,CP000001,F0005,2026-05-01,2026-05-22,2026-05-25,Mercadorias,12165.76,12165.76,Pago,Boleto
1,CP000002,F0002,2026-03-02,2026-04-01,NaN,Tecnologia,16428.88,0.00,Em aberto,PIX
2,CP000003,F0014,2026-03-30,2026-04-13,2026-04-12,Aluguel,19536.09,19536.09,Pago,Boleto
3,CP000004,F0022,2026-05-02,2026-06-16,NaN,Marketing,13913.86,0.00,Em aberto,Transferência
4,CP000005,F0019,2026-05-25,2026-06-15,NaN,Aluguel,12418.04,0.00,Em aberto,Boleto


In [8]:
df.tail()

,id_titulo_pagar,id_fornecedor,data_emissao,data_vencimento,data_pagamento,categoria_despesa,valor_titulo,valor_pago,status,forma_pagamento
399,CP000266,F0024,2026-03-28,2026-04-25,2026/04/27,Mercadorias,15682.59,15682.59,Pago,Boleto
400,CP000066,F0021,2026-07-27,2026-08-26,2026-08-30,Serviços,12909.99,12909.99,Pago,Transferência
401,CP000121,F0017,2026-04-16,2026-05-31,2026-05-29,Serviços,3961.71,3961.71,Pago,Transferência
402,CP000133,F0024,2026-06-01,2026-06-15,2026-06-18,Frete,14956.20,14956.20,Pago,Transferência
403,CP000379,F0002,2026-08-18,2026-08-25,2026-08-24,NaN,11610.80,11610.80,Pago,PIX


In [9]:
df.sample(5)

,id_titulo_pagar,id_fornecedor,data_emissao,data_vencimento,data_pagamento,categoria_despesa,valor_titulo,valor_pago,status,forma_pagamento
291,CP000292,F0010,2026-04-07,2026-04-21,2026-04-22,Energia,11510.14,11510.14,Pago,Débito automático
176,CP000177,F0016,2026-06-01,2026-06-15,NaN,NaN,5366.46,0.00,Em aberto,Boleto
13,CP000014,F0011,2026-02-23,2026-03-16,2026-03-19,Serviços,9260.02,9260.02,Pago,Boleto
331,CP000332,f0003,2026-01-20,2026-02-17,2026-02-21,Mercadorias,16931.32,16931.32,Pago,Débito automático
240,CP000241,f0025,2026-01-22,2026-02-12,NaN,Mercadorias,3067.05,0.00,Vencido,PIX


In [10]:
df.isnull().sum()

id_titulo_pagar        0
id_fornecedor          7
data_emissao           0
data_vencimento        0
data_pagamento       151
categoria_despesa      8
valor_titulo           7
valor_pago             7
status                 7
forma_pagamento        7
dtype: int64

In [11]:
q = '''SELECT id_titulo_pagar
            , id_fornecedor
            , data_pagamento
            , categoria_despesa
            , valor_titulo
            , valor_pago
            , status
            , forma_pagamento
        FROM df
        WHERE id_fornecedor isnull
            or data_pagamento isnull
            or categoria_despesa isnull
            or valor_titulo isnull
            or valor_pago isnull
            or status isnull
            or forma_pagamento isnull

'''

In [12]:
pysqldf(q)

,id_titulo_pagar,id_fornecedor,data_pagamento,categoria_despesa,valor_titulo,valor_pago,status,forma_pagamento
0,CP000002,F0002,NaN,Tecnologia,16428.88,0.0,Em aberto,PIX
1,CP000004,F0022,NaN,Marketing,13913.86,0.0,Em aberto,Transferência
2,CP000005,F0019,NaN,Aluguel,12418.04,0.0,Em aberto,Boleto
3,CP000006,F0004,NaN,Frete,18233.22,0.0,Em aberto,PIX
4,CP000007,F0021,NaN,Frete,4797.91,0.0,Vencido,Boleto
...,...,...,...,...,...,...,...,...
174,CP000395,F0009,NaN,Marketing,13656.33,0.0,Em aberto,Boleto
175,CP000396,F0009,NaN,Marketing,17787.90,0.0,Em aberto,Débito automático
176,CP000397,F0009,NaN,Serviços,1061.72,0.0,Em aberto,Transferência
177,CP000398,F0020,NaN,Tecnologia,15957.82,0.0,Vencido,Transferência


In [14]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 404 entries, 0 to 403
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   id_titulo_pagar    404 non-null    str    
 1   id_fornecedor      397 non-null    str    
 2   data_emissao       404 non-null    str    
 3   data_vencimento    404 non-null    str    
 4   data_pagamento     253 non-null    str    
 5   categoria_despesa  396 non-null    str    
 6   valor_titulo       397 non-null    float64
 7   valor_pago         397 non-null    float64
 8   status             397 non-null    str    
 9   forma_pagamento    397 non-null    str    
dtypes: float64(2), str(8)
memory usage: 31.7 KB


## Tratamento de dados

In [15]:
df_original = df.copy()

In [16]:
df = df.fillna('NAO INFORMADO')

In [17]:
df[['valor_titulo', 'valor_pago']] = df[['valor_titulo', 'valor_pago']].replace('NAO INFORMADO', 0)